In [10]:
import customtkinter as ctk
from tkinter import ttk, messagebox
from openpyxl import Workbook, load_workbook
import os

# ================= CONFIG =================
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

ADMIN_USERNAME = "branded4areason"
ADMIN_PASSWORD = "1718"

# ================= EXCEL MANAGER =================
class ExcelManager:
    @staticmethod
    def create(file, headers):
        if not os.path.exists(file):
            wb = Workbook()
            ws = wb.active
            ws.append(headers)
            wb.save(file)

    @staticmethod
    def read(file):
        if not os.path.exists(file):
            return [], []

        wb = load_workbook(file)
        ws = wb.active

        headers = [cell.value for cell in ws[1]]
        data = [(i,) + row for i, row in enumerate(
            ws.iter_rows(min_row=2, values_only=True), start=2)]

        return headers, data

    @staticmethod
    def append(file, row):
        wb = load_workbook(file)
        ws = wb.active
        ws.append(row)
        wb.save(file)

    @staticmethod
    def delete_by_value(file, col_index, value):
        wb = load_workbook(file)
        ws = wb.active

        for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
            if str(row[col_index].value) == str(value):
                ws.delete_rows(i)
                wb.save(file)
                return True

        return False


# ================= MAIN APP =================
class App:
    def __init__(self):
        self.root = ctk.CTk()
        self.root.title("BrandeD Fee  Pro")
        self.root.geometry("1100x600")

        self.start_menu()
        self.root.mainloop()

    def clear(self):
        for w in self.root.winfo_children():
            w.destroy()

    def exit_app(self):
        if messagebox.askyesno("Exit", "Are you sure?"):
            self.root.destroy()

    # ================= START =================
    def start_menu(self):
        self.clear()

        frame = ctk.CTkFrame(self.root, fg_color="transparent")
        frame.pack(expand=True)

        ctk.CTkLabel(frame,
                     text="BrandeD Fee Management System",
                     font=("Arial", 32, "bold")).pack(pady=30)

        ctk.CTkButton(frame, text="Login",
                      width=200,
                      command=self.login_screen).pack(pady=10)

        ctk.CTkButton(frame, text="Exit",
                      width=200,
                      fg_color="red",
                      command=self.exit_app).pack(pady=10)

    # ================= GLASS LOGIN =================
    def login_screen(self):
        self.clear()

        bg = ctk.CTkFrame(self.root, fg_color="#1a1a1a")
        bg.pack(fill="both", expand=True)

        container = ctk.CTkFrame(bg, fg_color="transparent")
        container.place(relx=0.5, rely=0.5, anchor="center")

        card = ctk.CTkFrame(
            container,
            width=360,
            height=420,
            corner_radius=20,
            fg_color="#2b2b2b"
        )
        card.pack()
        card.pack_propagate(False)

        ctk.CTkLabel(card,
                     text="BrandeD Fee System",
                     font=("Arial", 30, "bold")).pack(pady=(30, 5))

        ctk.CTkLabel(card,
                     text="BrandeD Login",
                     font=("Arial", 16),
                     text_color="#aaaaaa").pack(pady=(0, 20))

        user = ctk.CTkEntry(card, placeholder_text="Username",
                            width=260, height=40, corner_radius=12)
        user.pack(pady=10)

        pwd = ctk.CTkEntry(card, placeholder_text="Password",
                           show="*", width=260, height=40, corner_radius=12)
        pwd.pack(pady=10)

        def login():
            if user.get() == ADMIN_USERNAME and pwd.get() == ADMIN_PASSWORD:
                self.dashboard()
            else:
                messagebox.showerror("Error", "Invalid Credentials")

        ctk.CTkButton(card, text="Login",
                      width=260, height=42,
                      corner_radius=12,
                      command=login).pack(pady=20)

        ctk.CTkButton(card, text="Back",
                      width=260, height=40,
                      corner_radius=12,
                      fg_color="#444",
                      command=self.start_menu).pack()

    # ================= DASHBOARD =================
    def dashboard(self):
        self.clear()

        self.sidebar = ctk.CTkFrame(self.root, width=220)
        self.sidebar.pack(side="left", fill="y")

        self.content = ctk.CTkFrame(self.root)
        self.content.pack(side="right", expand=True, fill="both")

        ctk.CTkLabel(self.sidebar, text="MENU",
                     font=("Arial", 18, "bold")).pack(pady=20)

        self.nav("Add Student",
                 lambda: self.form(["Name","Age","Gender","Roll","Class"], "students.xlsx"))
        self.nav("View Students",
                 lambda: self.view("students.xlsx"))
        self.nav("Delete Student",
                 self.delete_student)

        self.nav("Add Fee", self.add_fee)
        self.nav("Pay Fee", self.pay_fee)
        self.nav("View Fees",
                 lambda: self.view("fees.xlsx"))

        ctk.CTkButton(self.sidebar, text="Logout",
                      command=self.start_menu).pack(side="bottom", pady=5)

        ctk.CTkButton(self.sidebar, text="Exit",
                      fg_color="red",
                      command=self.exit_app).pack(side="bottom", pady=5)

    def nav(self, text, cmd):
        ctk.CTkButton(self.sidebar, text=text,
                      width=180, command=cmd).pack(pady=5)

    def clear_content(self):
        for w in self.content.winfo_children():
            w.destroy()

    # ================= FORM =================
    def form(self, fields, file):
        self.clear_content()
        ExcelManager.create(file, fields)

        entries = {}

        for f in fields:
            ctk.CTkLabel(self.content, text=f).pack()
            e = ctk.CTkEntry(self.content)
            e.pack()
            entries[f] = e

        def save():
            values = [entries[f].get() for f in fields]

            if "" in values:
                messagebox.showerror("Error", "All fields required")
                return

            ExcelManager.append(file, values)
            messagebox.showinfo("Saved", "Data Saved")

        ctk.CTkButton(self.content, text="Save",
                      command=save).pack(pady=10)

    # ================= VIEW =================
    def view(self, file):
        self.clear_content()
        headers, data = ExcelManager.read(file)

        if not headers:
            messagebox.showerror("Error", "No file")
            return

        tree = ttk.Treeview(self.content)
        tree.pack(fill="both", expand=True)

        tree["columns"] = ["ID"] + headers
        tree["show"] = "headings"

        for col in ["ID"] + headers:
            tree.heading(col, text=col)

        for row in data:
            tree.insert("", "end", values=row)

    # ================= DELETE =================
    def delete_student(self):
        self.clear_content()

        entry = ctk.CTkEntry(self.content, placeholder_text="Roll Number")
        entry.pack(pady=20)

        def delete():
            success = ExcelManager.delete_by_value("students.xlsx", 3, entry.get())

            if success:
                messagebox.showinfo("Deleted", "Student removed")
            else:
                messagebox.showerror("Error", "Not found")

        ctk.CTkButton(self.content, text="Delete",
                      fg_color="red",
                      command=delete).pack()

    # ================= ADD FEE =================
    def add_fee(self):
        self.clear_content()

        fields = ["Name","Roll","Total Fee","Paid"]
        file = "fees.xlsx"

        ExcelManager.create(file,
            ["Name","Roll","Total Fee","Paid","Balance","Status"])

        entries = {}

        for f in fields:
            ctk.CTkLabel(self.content, text=f).pack()
            e = ctk.CTkEntry(self.content)
            e.pack()
            entries[f] = e

        def save():
            try:
                total = float(entries["Total Fee"].get())
                paid = float(entries["Paid"].get())

                balance = total - paid
                status = "Paid" if balance == 0 else "Partial"

                ExcelManager.append(file, [
                    entries["Name"].get(),
                    entries["Roll"].get(),
                    total, paid, balance, status
                ])

                messagebox.showinfo("Saved", "Fee Added")

            except:
                messagebox.showerror("Error", "Invalid Data")

        ctk.CTkButton(self.content, text="Save Fee",
                      command=save).pack(pady=10)

    # ================= PAY FEE =================
    def pay_fee(self):
        self.clear_content()

        roll = ctk.CTkEntry(self.content, placeholder_text="Roll")
        roll.pack(pady=10)

        amount = ctk.CTkEntry(self.content, placeholder_text="Amount")
        amount.pack(pady=10)

        def pay():
            try:
                wb = load_workbook("fees.xlsx")
                ws = wb.active

                for row in ws.iter_rows(min_row=2):
                    if row[1].value == roll.get():
                        amt = float(amount.get())

                        if amt > row[4].value:
                            messagebox.showerror("Error", "Too much")
                            return

                        row[3].value += amt
                        row[4].value -= amt
                        row[5].value = "Paid" if row[4].value == 0 else "Partial"

                        wb.save("fees.xlsx")
                        messagebox.showinfo("Success", "Payment Done")
                        return

                messagebox.showerror("Error", "Student not found")

            except:
                messagebox.showerror("Error", "Invalid Input")

        ctk.CTkButton(self.content, text="Pay Fee",
                      command=pay).pack(pady=20)


# ================= RUN =================
App()